# Example 5.3 / 1: Building and Cleaning Labels

Companion notebook for **5.3 Strings** in the AMPL lab Documentation, section "Building and Cleaning Labels". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Building and Cleaning Labels

The example creates a display label, extracts a product prefix, and replaces every run of
spaces with an underscore. The parameter `raw_name` is `symbolic`, so its values may be strings.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                        # Products
param raw_name {PROD} symbolic;  # Display name, may contain extra spaces
param code {PROD} symbolic;      # Product code such as BD-01

param label {p in PROD} symbolic = code[p] & ": " & raw_name[p];  # Code and name joined
param prefix {p in PROD} symbolic = substr(code[p], 1, 2);  # First two characters of the code
param slug {p in PROD} symbolic = gsub(raw_name[p], " +", "_");  # Name with each run of spaces as _
# 1 if the name contains Steel or steel
param has_steel {p in PROD} = match(raw_name[p], "[Ss]teel") > 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["raw_name"] = {"bands": "Steel bands", "coils": "Steel  coils"}
ampl.param["code"] = {"bands": "BD-01", "coils": "CL-02"}

In [5]:
# run.py
labels = ampl.param["label"].to_dict()
prefixes = ampl.param["prefix"].to_dict()
slugs = ampl.param["slug"].to_dict()
matches = ampl.param["has_steel"].to_dict()
for product in ampl.set["PROD"].to_list():
    label = labels[product]
    prefix = prefixes[product]
    slug = slugs[product]
    found = int(matches[product])
    print(f"{product}: {label} | {prefix} | {slug} | steel={found}")

bands: BD-01: Steel bands | BD | Steel_bands | steel=1
coils: CL-02: Steel  coils | CL | Steel_coils | steel=1


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param: raw_name code :=
    bands  "Steel bands"   "BD-01"
    coils  "Steel  coils"  "CL-02";

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
for {p in PROD} {
    printf "%s: %s | %s | %s | steel=%d\n",
        p, label[p], prefix[p], slug[p], has_steel[p];
}

bands: BD-01: Steel bands | BD | Steel_bands | steel=1
coils: CL-02: Steel  coils | CL | Steel_coils | steel=1
